# Faiss (Facebook AI Similarity Search)

This notebook focuses on the **Similarity Search** pipeline. 

We utilize [Faiss](https://ai.meta.com/tools/faiss/) to search the embeddings extracted from the ViT model and classify the multi-label test images.

![diagram](../images/pytorch-webinar-diagram.png)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from plantclef.config import get_device

print(f"PyTorch Version: {torch.__version__}")
device = get_device()
print(f"Using device: {device}")

In [ ]:
import pandas as pd
from pathlib import Path

# Get list of stored filed in cloud bucket
root = Path().resolve().parents[0]
print(root)
! date

In [ ]:
# path to data
data_path = f"{root}/data/embeddings"
train_path = f"{data_path}/train_embeddings"
test_path = f"{data_path}/test_grid_3x3_embeddings"

# read train/test data
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

# display data
print(f"Train DF shape: {train_df.shape}")
print(f"Test DF shape: {test_df.shape}")
display(train_df.head(3))
display(test_df.head(3))

## similarity search using FAISS

In [ ]:
import numpy as np
from plantclef.faiss.classifier import FaissClassifier

# similarity seearch using FAISS
nn_classifier = FaissClassifier(train_df)

# convert test embeddings to torch tensor
embs_array = np.array(test_df["embeddings"].tolist(), dtype=np.float32)
query_embs = torch.tensor(embs_array, device=get_device())
preds, similarities = nn_classifier.make_prediction(query_embs, k=1)
preds.shape, similarities.shape

In [ ]:
preds[:5], similarities[:5]

In [ ]:
# top 5 predictions for each embedding
preds, similarities = nn_classifier.make_prediction(query_embs, k=5)
preds[:2], similarities[:2]

In [ ]:
preds.shape, similarities.shape

### create dataframe with FAISS classifications

In [ ]:
# create dataframe with faiss classifications
def create_classification_dataframe(
    test_df: pd.DataFrame,
    predictions: np.array,
    similarities: np.array,
):
    cls_test_df = test_df.copy()
    cls_test_df["predictions"] = predictions.tolist()
    cls_test_df["similarities"] = similarities.tolist()
    return cls_test_df


faiss_df = create_classification_dataframe(test_df, preds, similarities)
faiss_df.head()

In [ ]:
from plantclef.plotting import plot_image_tiles

# select images from test set
image_names = ["CBN-Pyr-03-20230706.jpg", "CBN-can-E6-20230706.jpg"]
faiss_image_df = faiss_df[test_df["image_name"].isin(image_names)]

# show image tiles
plot_image_tiles(
    faiss_image_df,
    data_col="data",
    grid_size=3,
)

In [ ]:
from plantclef.plotting import plot_faiss_classifications

# plot tiles and image classifications from faiss
plot_faiss_classifications(
    train_df,
    faiss_image_df,
    grid_size=3,
    figsize=(20, 30),
)